**## Data-Level Security Using Row Filters and Column Masks**

This demonstrates how data-level security can be implemented directly on a table using row filters and column masks.

Unlike dynamic views, where logic is defined  within a view, this approach applies security rules directly to the table. These rules are enforced automatically whenever the table is queried.

The implementation covers two capabilities:

- Row-level Security: Control which rows are visible to a user
- Column-Level Masking: Control how sensitive values are displayed

A simple sales dataset is used to illustrate how different users receive different results when querying the same table.

**Step 1: Set up the schema for the table**

In [0]:
USE CATALOG demo;
USE SCHEMA data_security;

**Step 2: Create the table**

A table is defined to represent sales data across multiple regions.

In [0]:
DROP TABLE IF EXISTS sales_sesured;
CREATE OR REPLACE TABLE sales_secured (
    id INT,
    region STRING,
    email STRING,
    revenue INT
);

**Step 3: Insert Sample Data**

In [0]:
INSERT INTO sales_secured VALUES
(1, 'UK', 'john.doe@email.com', 10000),
(2, 'US', 'mike.smith@email.com', 20000),
(3, 'UK', 'sara.jones@email.com', 15000),
(4, 'US', 'david.brown@email.com', 25000),
(5, 'UK', 'emma.wilson@email.com', 18000);

In [0]:
SELECT * FROM sales_secured;

**Step 4:  Define the Row Function**

The function evaluates the user's group membership and returns a condition that determines whether a row should be included in the result.

For example:

- Users in the UK group are allowed to view the UK records
- Users in the US group are allowed to view the US records
- Admin users are allowed to view all the records


In [0]:
CREATE OR REPLACE FUNCTION fn_filter_region (region STRING)
RETURN
    is_account_group_member('admin-sg')
    OR (is_account_group_member('uk-sg') AND region = 'UK')
    OR (is_account_group_member('us-sg') AND region = 'US');

**Step 5: Apply Row Filter to Table** 

The row filter function is attached to the table. Once applied, the function is evaluated automatically at query time, and only the rows that satisfy the condition are returned.

In [0]:
ALTER TABLE sales_secured
SET ROW FILTER fn_filter_region ON (region);

In [0]:
DESC EXTENDED sales_secured;

In [0]:
SELECT * FROM sales_secured;

**STEP 6: Define Column Mask Function**

The function evaluates the user's group membership and determines whether to return the original value or a masked version.

For example:

- Admin users are allowed to view all the records
- Other users see a masked version of the email

In [0]:
CREATE OR REPLACE FUNCTION fn_mask_email (email STRING)
RETURNS STRING
RETURN
    CASE
        WHEN is_account_group_member('admin-sg') THEN email
        ELSE concat(substr(email, 1, 1), '***@', split(email,'@')[1])
       END

**Step 7: Apply Column Mask**

The masking function is attached to the email column. Once applied, the function is executed automatically whenever the column is queried

In [0]:
ALTER TABLE sales_secured
ALTER COLUMN email SET MASK fn_mask_email;


In [0]:
DESC EXTENDED sales_secured;

In [0]:
SELECT * FROM sales_secured;